# chess-vi — GRPO (RLVR) trên Colab

Notebook này **chỉ gọi script** `chessvi.train.grpo`. Reward nằm ở
`chessvi/train/reward.py` và có unit test riêng — chạy `pytest tests/test_reward.py`
trước khi tốn CU.

**Chốt chặn trước khi chạy:** SFT (T8) phải đạt accuracy 30–38% trên test set.
Dưới 20% là dữ liệu có vấn đề — quay lại T5, đừng chạy RL.

Rollout bằng **vLLM**. Cần GPU ≥ 24GB (A100/L4) cho model 4B; T4 16GB thì phải
giảm `--num-generations` và `--max-completion-length`.

## 1. Mount Drive + lấy repo

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/chessvi"
REPO_URL = "https://github.com/trantrien1/ChessVi.git"
REPO_DIR = "/content/ChessVi"

import os

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull --ff-only || true

## 2. Cài đặt

Cell này **không** cài `vllm`: smoke test ở cell 6 chạy `--no-vllm`, còn vLLM
ghim `torch==2.13.0` nên sẽ tải lại vài GB torch. Cài nó ở cell 7, khi thật sự
cần rollout.

Nếu thấy `ResolutionImpossible`, đừng chạy tiếp — dán nguyên khối ERROR ra.
Dòng `chessvi OK:` ở cuối là điều kiện để sang cell sau.

In [ ]:
# CHỈ cài train+data ở đây. Cell 6 (smoke test) chạy --no-vllm nên chưa cần
# vLLM, mà vLLM còn ghim torch==2.13.0 (vài GB, cài lại rất lâu). Để tới cell 7.
# Bỏ -q: lần chạy trước -q nuốt mất chi tiết, chỉ còn trơ "ResolutionImpossible".
!pip install -e ".[train,data]" 2>&1 | tail -30

# pip thất bại KHÔNG làm notebook dừng lại — lần trước cả ba cell sau vẫn chạy
# tiếp rồi chết vì ModuleNotFoundError, che mất nguyên nhân thật. Chốt ở đây.
!python -c "import chessvi; print('chessvi OK:', chessvi.__file__)" || echo ">>> CÀI ĐẶT HỎNG — DỪNG LẠI, ĐỪNG CHẠY CELL SAU"

## 3. Token + kiểm tra GPU

In [ ]:
import os

import torch

# Colab Secrets chỉ đọc được khi chạy từ giao diện web colab.research.google.com:
# userdata.get() hỏi ngược về frontend trong trình duyệt. Chạy từ VS Code hay một
# frontend khác thì không ai trả lời -> TimeoutException. Bắt rộng vì mỗi trường
# hợp hỏng một kiểu (ImportError, TimeoutException, SecretNotFoundError).
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata

        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception as error:
        print(f"Không lấy được Colab Secret ({type(error).__name__}).")
        import getpass

        # getpass không in token ra output, nên notebook commit lên không lộ.
        os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN (Enter để bỏ qua): ")

# Token CHỈ cần khi push lên Hub (cell 7/8). Cell 4-6 chạy được mà không có.
if not os.environ.get("HF_TOKEN"):
    print("CHƯA CÓ HF_TOKEN — cell 4-6 vẫn chạy, cell 7/8 sẽ không push được.")

!nvidia-smi
print("CUDA:", torch.cuda.is_available())

## 4. Test hàm reward (không cần GPU)

Reward sai dấu thì RL vẫn chạy mà model càng học càng tệ. Luôn chạy cell này trước.

In [ ]:
!pip install -q pytest==9.1.1
!python -m pytest tests/test_reward.py tests/test_grpo.py -q

## 5. Chuẩn bị puzzle set (nếu chưa có)

In [ ]:
PUZZLE_DIR = f"{DRIVE_ROOT}/data/puzzles"

import os

if not os.path.exists(f"{PUZZLE_DIR}/train.parquet"):
    !python -m chessvi.data.puzzles --train-size 50000 --test-size 2000 --out-dir {PUZZLE_DIR}
!ls -la {PUZZLE_DIR}

## 6. Smoke test

5 step, model 0.6B, tắt vLLM. Phải chạy xong không lỗi trước khi chạy thật.

In [ ]:
!python -m chessvi.train.grpo \
    --puzzles {PUZZLE_DIR}/train.parquet \
    --base-model Qwen/Qwen3-0.6B \
    --limit 16 --max-steps 5 \
    --batch-size 2 --grad-accum 1 --num-generations 2 --max-completion-length 64 \
    --no-vllm --no-push \
    --output-dir /content/outputs/grpo-smoke

## 7. Chạy thật

`--adapter` trỏ vào adapter LoRA đã SFT ở T8: RL tiếp tục từ đó chứ không
bắt đầu lại từ base model.

Cell này cài `vllm` trước. **Lưu ý:** `vllm==0.28.0` ghim `torch==2.13.0`, nên
nếu Colab đang có torch bản khác thì pip sẽ tải lại torch (vài GB) và runtime
có thể cần restart. Cài xong mà `import torch` báo lỗi CUDA thì Runtime →
Restart rồi chạy lại cell 1–3.

In [ ]:
!pip install -e ".[rl]" 2>&1 | tail -20

SFT_ADAPTER = f"{DRIVE_ROOT}/outputs/sft"
HUB_MODEL_ID = "trantrien1/chessvi-4b-grpo"
OUTPUT_DIR = f"{DRIVE_ROOT}/outputs/grpo"

!python -m chessvi.train.grpo \
    --puzzles {PUZZLE_DIR}/train.parquet \
    --base-model Qwen/Qwen3-4B \
    --adapter {SFT_ADAPTER} \
    --output-dir {OUTPUT_DIR} \
    --hub-model-id {HUB_MODEL_ID} \
    --num-generations 8 --batch-size 8 --grad-accum 4 --save-steps 100

## 8. Resume sau khi Colab ngắt

In [ ]:
!python -m chessvi.train.grpo \
    --puzzles {PUZZLE_DIR}/train.parquet \
    --base-model Qwen/Qwen3-4B \
    --adapter {SFT_ADAPTER} \
    --output-dir {OUTPUT_DIR} \
    --hub-model-id {HUB_MODEL_ID} \
    --num-generations 8 --batch-size 8 --grad-accum 4 --save-steps 100 \
    --resume

## 9. Chốt chặn sau T9

RL phải cải thiện **≥ 4 điểm** so với SFT. Không đạt thì cấu hình reward sai —
soi `rewards/puzzle_reward/mean` trong log: nếu nó dính sát -1 thì model đang
không sinh ra nước hợp lệ, kiểm tra lại prompt và `max_completion_length`.